What it is: Entity Memory extracts named entities from conversation and maintains a living knowledge base of facts about each one.

When you need it: Your agent must track and update structured information about specific people, orgs, or projects over time.

The trade-off: Entity extraction is imperfect, and merging or disambiguating entities with similar names is a hard problem.

Imagine a personal assistant who keeps a contact card for every person, place, and project you mention. Each time you share new info ("Sarah got promoted"), they update the right card. Entity Memory gives your LLM agent that same ability. This agent memory technique uses entity extraction (spotting names of people, places, and things in text) to build a structured knowledge base. The agent creates or updates records as new facts appear, so it never asks the same question twice. It's especially useful for personal assistant chatbots and customer support agents that track user details across sessions.

Entity Memory works the same way. Instead of storing raw message history, it extracts structured information about named things (people, organizations, projects, preferences) and maintains a living knowledge base. When the user mentions "my manager Sarah" or "the Q3 budget project," the system creates or updates a record with the relevant details.

On later turns, the agent retrieves entity records and injects them into the prompt. This gives the agent persistent, structured knowledge about the people and things the user cares about. It's especially useful for personal assistant and CRM-style agents.

Keywords: agent memory, LLM agent, entity memory, entity extraction, named entity recognition, structured knowledge base, personal assistant, customer support, cross-session memory

### Key Concepts

Entity extraction (NER): Using named entity recognition (through LLM prompts or NER models) to identify entities in each message: people, places, organizations, dates, concepts. NER stands for Named Entity Recognition, the task of spotting proper nouns and classifying them.

Entity schemas: Defining what attributes to track per entity type. A person entity might have fields like name, role, relationship, and last-mentioned date. A project entity might have status, deadline, and stakeholders.

Entity updates and merging: When new information about an existing entity appears, the system updates the record. Merging logic handles conflicts (e.g., a corrected phone number) and disambiguation (e.g., two people named "Alex").

Entity-context injection: Before each LLM call, the system retrieves records for entities mentioned in the current message. It injects them into the prompt as structured context.

Structured vs. unstructured entity stores: You can store entity information as structured records (JSON, database rows) or as free-text descriptions. Structured stores enable precise queries. Unstructured stores are more flexible.

> **Where we are in the progression:**
>
> | Technique | What it stores | How it retrieves | Gap it leaves |
> |---|---|---|---|
> | 06 — Vector Store | Raw message text as vectors | Semantic similarity search | Finds meaning but not structure — returns full sentences, not clean facts |
> | **07 — Entity Memory** | **Structured facts about named entities** | **Direct key lookup by entity name** | **No relationships between entities — who owns what?** |
>
> Entity Memory answers the question Technique 06 left open: when you want to know "what is Chiru's salary", you should not have to run a semantic search and parse the result. You should get back `{"salary": "₹1,20,000"}` — immediately, precisely, always current.

## What Is Entity Memory?

### The core idea in one sentence
Extract structured facts about named entities (people, accounts, goals, assets) from the conversation, store them as a profile record, and **update them in-place** when new information arrives — so the profile always reflects the current state.

---

### The mental model — a CRM record that updates itself

A CRM (Customer Relationship Management) system stores structured facts about each customer: name, contact details, company, preferences. When a customer calls and says "I changed jobs", the support agent updates their record. The old job is replaced. The record always reflects the current state.

Entity Memory is a CRM for your agent — but it updates itself automatically, extracting facts from conversation without human intervention.

```
ENTITY PROFILE for user: chiru_001
──────────────────────────────────────────────────────
personal:
  name:            Chiru
  age:             32
  location:        Mumbai

financial:
  monthly_salary:  ₹1,20,000
  monthly_expenses: ₹60,000
  monthly_surplus:  ₹60,000
  risk_profile:    conservative

assets:
  fd_amount:       ₹50,000
  fd_maturity:     3 months

goals:
  retirement_age:  55
  emergency_fund_target: 6 months expenses
──────────────────────────────────────────────────────
last_updated: 2025-06-12T10:30:00Z
```

This profile is injected into every API call. The model always has structured, current facts — without running a search.

---

### How it differs from Technique 06

| | Vector Store (06) | Entity Memory (07) |
|---|---|---|
| Storage unit | Raw message text | Structured key-value facts |
| Retrieval | Semantic similarity search | Direct key lookup |
| Update model | Append-only | **In-place update** — old value replaced |
| Stale facts | ❌ Old and new both retrieved | ✅ Only current value stored |
| Query | "find messages about salary" | `profile["salary"]` |
| Token cost | Variable (depends on search results) | Fixed (profile size) |
| Best for | Fuzzy recall, open-ended retrieval | Structured facts that change over time |

---

### Point 1 — Extraction: the LLM reads the conversation and fills the profile

After each turn, we call GPT-4o with a structured extraction prompt:

> *"Read this message and extract any facts about the user. Return a JSON object with the extracted fields. Only return fields that were explicitly mentioned."*

GPT-4o returns something like:
```json
{"monthly_salary": "₹1,20,000", "age": 32}
```

We merge this into the profile — new fields are added, existing fields are updated.

---

### Point 2 — Update model: merge, not append

When the user says "I got a raise — my new salary is ₹1,50,000", we do not add a second salary entry alongside the first. We **replace** the old value with the new one. The profile always shows the current state.

This is the fundamental improvement over a vector store: no stale facts, no retrieval ambiguity, no model confusion about which salary is current.

---

### Point 3 — The profile is always injected — not retrieved

Unlike vector store memory (which retrieves based on query relevance), the entity profile is **always** injected into every API call in full. It is a constant context block. This means the model always has the full user profile, regardless of what the current query is about. No missed retrieval, no relevance threshold to tune.

The trade-off: profile tokens are always consumed, even when the query does not need them. For a compact profile (~200 tokens), this is a good trade.

---

### Point 4 — Schema design is a domain decision

The profile schema — what fields to track, how to name them, what types they hold — is the most important design decision in this technique. A bad schema loses information. A good schema captures everything the agent needs to give personalised advice.

For FinCoach, we define a schema with five categories: personal, financial, assets, goals, and preferences. Every field has a type, description, and default of `null` (meaning not yet known).

---

### Point 5 — Entity Memory and Vector Store Memory are complementary

They solve different problems:

- **Entity Memory** for structured, current, frequently-injected facts: salary, risk profile, goals
- **Vector Store** for unstructured, historical, search-retrieved context: past conversations, specific events, detailed discussions

Production systems use both together. The entity profile handles the "who is this user" question. The vector store handles the "what did we discuss" question.

---

## Trade-offs

| | |
|---|---|
| ✅ Always current — in-place updates, no stale facts | ❌ Schema must be pre-defined — novel facts may not fit |
| ✅ Direct lookup — no search needed | ❌ Extraction call adds cost and latency per turn |
| ✅ Fixed, predictable token cost per call | ❌ Extraction can hallucinate facts not in the conversation |
| ✅ Solves the stale fact problem of vector stores | ❌ No relationships between entities (→ Technique 08) |
| ✅ Injected always — no missed retrieval | ❌ Profile size grows with schema breadth |

## Production Verdict

> **Essential for any domain with structured, changing user facts. Always combine with vector store.**
>
> Entity Memory is the structured complement to vector store retrieval. In financial services, the user's salary, risk profile, and goals are queried on nearly every turn — they should be in a fast, structured, always-current profile, not buried in a semantic search index. Build this layer for any agent that serves returning users with evolving personal data.

---

In [2]:

# Install required packages.
# 'openai'   — GPT-4o for conversation AND entity extraction.
# 'tiktoken' — Token counting.
!pip install openai tiktoken --quiet

In [3]:

# --- Standard library ---
import json                              # Profile persistence and extraction parsing.
import os                                # Environment variables.
import time                              # Rate-limit delays.
from copy import deepcopy                # Deep copy for profile snapshots.
from datetime import datetime, timezone  # UTC timestamps.
from typing import List, Dict, Optional, Any  # Type hints.
from dataclasses import dataclass, field, asdict  # Data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.

In [4]:

# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid."

client    = openai.OpenAI(api_key=OPENAI_API_KEY)
# Used for both conversation calls AND entity extraction calls.

CHAT_MODEL       = "gpt-4o"
# Main conversation model.

EXTRACTION_MODEL = "gpt-4o-mini"
# Cheaper model for entity extraction.
# Extraction is a structured parsing task — mini is sufficient.
# 16x cheaper than gpt-4o at $0.15/M vs $2.50/M input tokens.

TOKENISER = tiktoken.get_encoding("o200k_base")

print(f"Chat model       : {CHAT_MODEL}")
print(f"Extraction model : {EXTRACTION_MODEL}")

Key loaded from environment variable.
Chat model       : gpt-4o
Extraction model : gpt-4o-mini


---
## Part 1 — The FinCoach Entity Schema

The schema defines every fact we want to track. This is the most important design decision in the technique. Every field has a name, type, description, and default of `None` (not yet known).

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# FINCOACH ENTITY SCHEMA
# Every field the agent wants to track about a user.
# Fields are grouped into categories for readability and injection control.
# ─────────────────────────────────────────────────────────────────────────────

FINCOACH_ENTITY_SCHEMA = {

    # ── Personal context ──────────────────────────────────────────────────────
    "name": {
        "type": "str",
        "description": "User's first name or preferred name",
        "category": "personal",
        "value": None,
        # None = not yet known. The agent should not guess.
    },
    "age": {
        "type": "int",
        "description": "User's current age in years",
        "category": "personal",
        "value": None,
    },
    "location": {
        "type": "str",
        "description": "City or region where the user lives",
        "category": "personal",
        "value": None,
    },
    "employer": {
        "type": "str",
        "description": "Current employer or company name",
        "category": "personal",
        "value": None,
        # This field demonstrates the update-in-place behaviour —
        # when the user changes jobs, this is replaced, not appended.
    },
    "dependents": {
        "type": "int",
        "description": "Number of financial dependents (family members)",
        "category": "personal",
        "value": None,
    },

    # ── Financial profile ─────────────────────────────────────────────────────
    "monthly_salary": {
        "type": "str",
        "description": "Monthly take-home salary after tax (include currency symbol)",
        "category": "financial",
        "value": None,
        # Stored as str to preserve the ₹ symbol and exact formatting.
    },
    "monthly_expenses": {
        "type": "str",
        "description": "Total monthly expenses (rent, food, transport, etc.)",
        "category": "financial",
        "value": None,
    },
    "monthly_surplus": {
        "type": "str",
        "description": "Monthly surplus = salary minus expenses",
        "category": "financial",
        "value": None,
        # Can be computed but we track it explicitly in case the user
        # states it directly with a different value (e.g. accounts for irregular expenses).
    },
    "risk_profile": {
        "type": "str",
        "description": "Investment risk tolerance: conservative / moderate / aggressive",
        "category": "financial",
        "value": None,
        # One of three values — validated during extraction.
    },
    "tax_regime": {
        "type": "str",
        "description": "Indian income tax regime: old / new",
        "category": "financial",
        "value": None,
    },

    # ── Assets ────────────────────────────────────────────────────────────────
    "fd_amount": {
        "type": "str",
        "description": "Fixed Deposit amount (include currency symbol)",
        "category": "assets",
        "value": None,
    },
    "fd_maturity_months": {
        "type": "int",
        "description": "Months until FD matures (approximate)",
        "category": "assets",
        "value": None,
    },
    "existing_sip_amount": {
        "type": "str",
        "description": "Monthly SIP investment amount if any",
        "category": "assets",
        "value": None,
    },
    "existing_investments": {
        "type": "str",
        "description": "Summary of other existing investments (PPF, stocks, property, etc.)",
        "category": "assets",
        "value": None,
    },

    # ── Goals ─────────────────────────────────────────────────────────────────
    "retirement_age": {
        "type": "int",
        "description": "Target retirement age",
        "category": "goals",
        "value": None,
    },
    "retirement_corpus_target": {
        "type": "str",
        "description": "Target retirement corpus amount",
        "category": "goals",
        "value": None,
    },
    "short_term_goal": {
        "type": "str",
        "description": "Primary short-term financial goal (within 1-3 years)",
        "category": "goals",
        "value": None,
    },
    "emergency_fund_target_months": {
        "type": "int",
        "description": "Target emergency fund size in months of expenses",
        "category": "goals",
        "value": None,
    },

    # ── Preferences ───────────────────────────────────────────────────────────
    "investment_constraints": {
        "type": "str",
        "description": "Hard constraints on investments (e.g. 'never equity', 'no crypto')",
        "category": "preferences",
        "value": None,
        # This is the field that must survive every compression cycle.
        # It is the most safety-critical fact in the entire schema.
    },
    "preferred_instruments": {
        "type": "str",
        "description": "Preferred investment instruments (FD, SIP, PPF, NPS, etc.)",
        "category": "preferences",
        "value": None,
    },
}

print(f"FinCoach entity schema defined: {len(FINCOACH_ENTITY_SCHEMA)} fields")
print(f"Categories: {set(v['category'] for v in FINCOACH_ENTITY_SCHEMA.values())}")

FinCoach entity schema defined: 20 fields
Categories: {'assets', 'financial', 'preferences', 'goals', 'personal'}


---
## Part 2 — The Entity Extractor

This component reads a message and returns a JSON object of extracted facts. Built as a standalone function — independently testable and swappable.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# The extraction prompt tells the LLM exactly what to extract and how.
# It is generated dynamically from the schema so adding a new field
# to the schema automatically updates the extraction prompt.
# ─────────────────────────────────────────────────────────────────────────────

def _build_extraction_prompt(schema: Dict) -> str:
    """
    Build the extraction system prompt from the entity schema.
    Called once at startup — result is cached in EXTRACTION_SYSTEM_PROMPT.
    """
    field_list = []
    for field_name, field_def in schema.items():
        field_list.append(
            f"  - \"{field_name}\" ({field_def['type']}): {field_def['description']}"
        )
        # Each field becomes a line: "field_name" (type): description

    fields_text = "\n".join(field_list)

    return f"""You are a financial data extraction assistant for FinCoach.
Read the user message and extract any facts that match the fields below.

FIELDS TO EXTRACT:
{fields_text}

STRICT RULES:
1. Return ONLY a valid JSON object — no markdown, no explanation, no preamble.
2. Include ONLY fields explicitly mentioned in the message — do NOT infer or guess.
3. If no relevant facts are present, return an empty object: {{}}
4. Use null for any field where the value is unclear or ambiguous.
5. Preserve currency symbols (₹) and units exactly as stated.
6. For risk_profile: only use 'conservative', 'moderate', or 'aggressive'.

Example input : "My salary is ₹1,20,000 and I never invest in equity."
Example output: {{"monthly_salary": "₹1,20,000", "investment_constraints": "never invest in equity"}}
"""


# Build once and cache — the prompt is the same for all extraction calls.
EXTRACTION_SYSTEM_PROMPT = _build_extraction_prompt(FINCOACH_ENTITY_SCHEMA)
print("Extraction system prompt built from schema.")


def extract_entities(message_content: str) -> Dict[str, Any]:
    """
    Extract structured entity facts from a single message.

    Args:
        message_content: The raw text of a user message.

    Returns:
        A dict of extracted field name → value pairs.
        Empty dict {} if no relevant facts were found.
        None values for fields that were mentioned but ambiguous.

    Production notes:
    - Run this on user messages only — assistant messages rarely contain new user facts.
    - The call is asynchronous in production — fire-and-forget after the API response.
    - Cache results by message hash to avoid re-extracting the same message.
    """

    try:
        response = client.chat.completions.create(
            model=EXTRACTION_MODEL,
            # gpt-4o-mini: 16x cheaper than gpt-4o for this structured task.

            max_tokens=400,
            # Extraction output is a compact JSON object — 400 tokens is generous.

            temperature=0.0,
            # Fully deterministic — fact extraction should not be creative.

            response_format={"type": "json_object"},
            # Force JSON output mode — OpenAI guarantees valid JSON in the response.
            # This eliminates the need to parse markdown code blocks or handle
            # the model saying 'Here is the JSON:' before the actual JSON.

            messages=[
                {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                # The domain-specific extraction instructions.
                {"role": "user",   "content": f"Extract facts from: {message_content}"},
                # The message to extract from.
            ]
        )

        raw_json = response.choices[0].message.content
        # The model's response — guaranteed to be valid JSON by response_format.

        extracted = json.loads(raw_json)
        # Parse the JSON string into a Python dict.

        # Filter to only known schema fields — ignore anything extra the model hallucinated.
        valid_fields = {k: v for k, v in extracted.items()
                        if k in FINCOACH_ENTITY_SCHEMA and v is not None}
        # k in FINCOACH_ENTITY_SCHEMA: only keep fields we defined.
        # v is not None: skip fields the model returned as null (nothing extracted).

        return valid_fields

    except (json.JSONDecodeError, Exception) as e:
        # If extraction fails for any reason, return empty — never crash the agent.
        # In production: log the error with the message hash for debugging.
        print(f"  [EXTRACT] Warning: extraction failed: {e}")
        return {}


# --- Quick test ---
test_msg = "My monthly salary is ₹1,20,000 and my expenses are ₹60,000. I never invest in equity."
test_extracted = extract_entities(test_msg)
print(f"\nTest extraction:")
print(f"  Input   : '{test_msg}'")
print(f"  Extracted: {json.dumps(test_extracted, ensure_ascii=False, indent=4)}")

Extraction system prompt built from schema.

Test extraction:
  Input   : 'My monthly salary is ₹1,20,000 and my expenses are ₹60,000. I never invest in equity.'
  Extracted: {
    "monthly_salary": "₹1,20,000",
    "monthly_expenses": "₹60,000",
    "investment_constraints": "never invest in equity"
}


---
## Part 3 — The Entity Profile Store

The profile store manages a single user's entity profile: initialisation from the schema, in-place updates from extraction results, persistence, and injection formatting.

In [7]:
class EntityProfile:
    """
    Manages a single user's structured entity profile.

    The profile is a flat dict of field_name → current_value.
    Fields start as None (unknown). They are filled in as the user shares information.
    When a field is updated, the old value is replaced — no history is kept here.
    (History is kept in the archive — see EntityMemory class.)
    """

    def __init__(self, user_id: str, schema: Dict):
        self.user_id = user_id
        # The user this profile belongs to.

        self._schema = schema
        # Reference to the schema — used for field validation and injection formatting.

        self._profile: Dict[str, Any] = {
            field_name: None
            for field_name in schema.keys()
        }
        # Initialise all fields to None — nothing is known yet.
        # Each field corresponds to exactly one schema entry.

        self._update_history: List[Dict] = []
        # Log of every update — field, old value, new value, timestamp.
        # Not injected into the API — used for auditing and debugging.
        # In financial services: this is your audit trail for advice given
        # based on what facts were known at what time.

        self.last_updated: Optional[str] = None
        # Timestamp of the most recent update — injected into the profile block.

    def update(self, extracted_facts: Dict[str, Any]) -> List[str]:
        """
        Merge extracted facts into the profile.
        Updates fields in-place — old values are replaced, not appended.

        Args:
            extracted_facts: Dict of field_name → new_value from the extractor.

        Returns:
            List of field names that were actually updated (for logging).
        """

        updated_fields = []

        for field_name, new_value in extracted_facts.items():

            if field_name not in self._profile:
                # Field not in schema — skip (already filtered in extract_entities,
                # but guard defensively here too).
                continue

            old_value = self._profile[field_name]
            # Record the old value before updating.

            if old_value != new_value:
                # Only record a history entry if the value actually changed.
                # If the same value is mentioned again, no update needed.

                self._profile[field_name] = new_value
                # IN-PLACE UPDATE: the old value is gone, replaced by the new one.
                # This is what prevents the stale fact problem:
                # there is always exactly one current value per field.

                self._update_history.append({
                    "field":     field_name,
                    "old_value": old_value,
                    "new_value": new_value,
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                })
                # Log: what changed, from what, to what, when.
                # In production: write this to a separate audit table in your DB.

                updated_fields.append(field_name)

        if updated_fields:
            self.last_updated = datetime.now(timezone.utc).isoformat()
            # Update the profile's last_updated timestamp.

        return updated_fields

    def get_known_fields(self) -> Dict[str, Any]:
        """
        Return only fields that have a non-None value.
        This is what gets injected into the API call — unknown fields are not injected
        (they would just say 'unknown: None' which wastes tokens and confuses the model).
        """
        return {k: v for k, v in self._profile.items() if v is not None}

    def format_for_injection(self) -> str:
        """
        Format the profile as a readable context block for injection into the API call.
        Groups fields by category for readability.
        Only includes known (non-None) fields.
        """
        known = self.get_known_fields()

        if not known:
            return ""  # Empty profile — nothing to inject yet.

        # Group by category.
        by_category: Dict[str, List[str]] = {}
        for field_name, value in known.items():
            category = self._schema[field_name]["category"]
            if category not in by_category:
                by_category[category] = []
            by_category[category].append(f"  {field_name}: {value}")

        # Format as a structured block.
        lines = [f"USER PROFILE for {self.user_id}:"]
        for category, field_lines in by_category.items():
            lines.append(f"\n[{category.upper()}]")
            lines.extend(field_lines)

        if self.last_updated:
            lines.append(f"\nlast_updated: {self.last_updated[:19]}Z")
            # Show only date and time (not full ISO string) for brevity.

        return "\n".join(lines)

    def token_count(self) -> int:
        """Count tokens the profile block will consume when injected."""
        return len(TOKENISER.encode(self.format_for_injection()))

    def completion_percentage(self) -> float:
        """What fraction of schema fields are currently known (non-None)?"""
        total  = len(self._profile)
        known  = sum(1 for v in self._profile.values() if v is not None)
        return known / total if total > 0 else 0.0

    def to_dict(self) -> Dict:
        """Serialise for persistence."""
        return {
            "user_id":        self.user_id,
            "profile":        self._profile,
            "update_history": self._update_history,
            "last_updated":   self.last_updated,
        }

    @classmethod
    def from_dict(cls, data: Dict, schema: Dict) -> "EntityProfile":
        """Restore from a serialised dict."""
        ep = cls(user_id=data["user_id"], schema=schema)
        ep._profile        = data["profile"]
        ep._update_history = data["update_history"]
        ep.last_updated    = data["last_updated"]
        return ep

    def print_profile(self) -> None:
        """Print a human-readable profile summary."""
        known = self.get_known_fields()
        print(f"\n{'='*55}")
        print(f"  Entity Profile — User: {self.user_id}")
        print(f"  Completion: {self.completion_percentage():.0%} "
              f"({len(known)}/{len(self._profile)} fields known)")
        print(f"  Profile tokens: ~{self.token_count()}")
        print(f"{'='*55}")
        print(self.format_for_injection() or "  (No fields known yet)")
        print(f"{'='*55}\n")


print("EntityProfile class defined.")

EntityProfile class defined.


---
## Part 4 — The EntityMemory Class

The memory class orchestrates everything: it wraps the entity profile, manages the recent conversation buffer, runs extraction after each user message, and builds the API context.

In [8]:
class EntityMemory:
    """
    Conversation memory augmented with an always-current entity profile.

    On every turn:
    1. Extract structured facts from the user message → update entity profile
    2. Build API context: [system] + [entity profile] + [recent buffer]
    3. The model sees the full, current, structured profile on every call

    No search needed. No stale facts. Profile is always current.
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_recent_turns: int = 5,
        # Recent turns to keep verbatim alongside the profile.
        extract_async: bool = False,
        # If True: extraction runs after the API response (async — no latency hit).
        # If False: extraction runs before the API call (profile updated each turn).
        # For this notebook: False — synchronous, so we can see updates turn-by-turn.
        # In production: True — fire extraction after the response, update the profile
        # before the next turn. Zero latency impact.
    ):
        self.session_id       = session_id
        self.user_id          = user_id
        self.system_prompt    = system_prompt
        self.max_recent_turns = max_recent_turns
        self.extract_async    = extract_async

        self.entity_profile = EntityProfile(
            user_id=user_id,
            schema=FINCOACH_ENTITY_SCHEMA
        )
        # The structured profile — starts empty, fills as the user shares information.

        self._recent_buffer: List[Dict] = []
        # Recent turns verbatim — for conversational coherence.

        self._archive: List[Dict] = []
        # Complete session history — for compliance and debugging.

        self._total_turns      = 0
        self._extraction_count = 0  # How many extraction calls were made.
        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[EntityMemory] Initialised — session: {self.session_id}")
        print(f"  User: {self.user_id} | Recent turns: {self.max_recent_turns}")

    def _run_extraction(self, user_message: str) -> None:
        """
        Extract entities from a user message and update the profile.
        Called after every user message.
        """
        extracted = extract_entities(user_message)
        # Call the extractor — returns a dict of field_name → value.

        if extracted:
            updated_fields = self.entity_profile.update(extracted)
            # Merge into the profile — in-place updates for changed fields.

            if updated_fields:
                print(f"  [PROFILE] Updated: {updated_fields}")
                # Log which fields changed — useful for debugging.
                for field in updated_fields:
                    new_val = self.entity_profile._profile[field]
                    print(f"    {field} → {new_val}")
        else:
            print(f"  [PROFILE] No new facts extracted")

        self._extraction_count += 1

    def add_message(self, role: str, content: str) -> None:
        """
        Add a message to the recent buffer and archive.
        For user messages: run entity extraction to update the profile.
        """
        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'.")

        msg = {"role": role, "content": content,
               "timestamp": datetime.now(timezone.utc).isoformat()}

        # Add to recent buffer — evict oldest if over limit.
        self._recent_buffer.append({"role": role, "content": content})
        if len(self._recent_buffer) > self.max_recent_turns * 2:
            self._recent_buffer.pop(0)

        self._archive.append(msg)
        # Full history — every message ever, never evicted.

        if role == "user" and not self.extract_async:
            # Synchronous extraction: run immediately after the user message.
            # The profile is updated BEFORE the API call.
            # This means the model sees the updated profile for THIS turn.
            self._run_extraction(content)

        if role == "assistant":
            self._total_turns += 1

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Build the complete API message list.

        Structure:
          [system: FinCoach persona]
          [system: user entity profile]  ← always injected, always current
          [recent buffer verbatim]
        """
        messages = []

        messages.append({"role": "system", "content": self.system_prompt})
        # 1. FinCoach's persona — always first.

        profile_text = self.entity_profile.format_for_injection()
        if profile_text:
            messages.append({
                "role": "system",
                "content": (
                    f"KNOWN USER FACTS (always current — use these to personalise advice):\n"
                    f"{profile_text}\n"
                    f"If a fact seems outdated, trust the most recent conversation turn."
                )
            })
            # 2. Entity profile — injected as system context.
            # The trailing instruction prevents the model from using a profile field
            # over something the user JUST said in the current turn.

        for msg in self._recent_buffer:
            messages.append(msg)
            # 3. Recent turns verbatim — current session context.

        return messages

    def persist(self, filepath: str) -> None:
        """Save the complete memory state to a JSON file."""
        record = {
            "schema_version":  "1.0",
            "technique":       "entity_memory",
            "session_id":      self.session_id,
            "user_id":         self.user_id,
            "created_at":      self.created_at,
            "saved_at":        datetime.now(timezone.utc).isoformat(),
            "config": {
                "max_recent_turns": self.max_recent_turns,
                "extract_async":    self.extract_async,
            },
            "stats": {
                "total_turns":      self._total_turns,
                "extraction_calls": self._extraction_count,
                "profile_completion": f"{self.entity_profile.completion_percentage():.0%}",
                "profile_tokens":   self.entity_profile.token_count(),
            },
            "entity_profile":  self.entity_profile.to_dict(),
            "recent_buffer":   self._recent_buffer,
            "archive":         self._archive,
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f"[EntityMemory] Persisted — profile {record['stats']['profile_completion']} complete")

    @classmethod
    def load(cls, filepath: str) -> "EntityMemory":
        """Restore EntityMemory from a saved JSON file."""
        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)
        mem = cls(
            session_id=record["session_id"],
            user_id=record["user_id"],
            system_prompt="[LOADED — set system_prompt after loading]",
            max_recent_turns=record["config"]["max_recent_turns"],
            extract_async=record["config"]["extract_async"],
        )
        mem.entity_profile   = EntityProfile.from_dict(
            record["entity_profile"], FINCOACH_ENTITY_SCHEMA
        )
        mem._recent_buffer   = record["recent_buffer"]
        mem._archive         = record["archive"]
        mem._total_turns     = record["stats"]["total_turns"]
        mem._extraction_count = record["stats"]["extraction_calls"]
        mem.created_at       = record["created_at"]
        print(f"[EntityMemory] Loaded — profile {record['stats']['profile_completion']} complete")
        return mem

    def print_stats(self) -> None:
        """Print memory and profile stats."""
        ctx_tokens = sum(
            len(TOKENISER.encode(m["content"])) for m in self.get_messages_for_api()
        )
        print(f"\n{'='*58}")
        print(f"  Entity Memory Stats — Session: {self.session_id}")
        print(f"{'='*58}")
        print(f"  Total turns          : {self._total_turns}")
        print(f"  Extraction calls     : {self._extraction_count}")
        print(f"  Profile completion   : {self.entity_profile.completion_percentage():.0%}")
        print(f"  Profile tokens       : ~{self.entity_profile.token_count()}")
        print(f"  Recent buffer turns  : {len(self._recent_buffer)//2}")
        print(f"  Total context tokens : ~{ctx_tokens}")
        print(f"{'='*58}")
        self.entity_profile.print_profile()


print("EntityMemory class defined.")

EntityMemory class defined.


---
## Part 5 — The FinCoach Agent

In [9]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Use the KNOWN USER FACTS profile to personalise every response.
- Reference specific numbers from the profile without asking the user to repeat themselves.
- If a fact seems to have changed in the current message, use the new value.
- If a critical fact is missing from the profile, ask for it once.
- Keep responses concise — 3 to 5 sentences unless asked for detail.
- Never recommend specific stocks.
- Always recommend consulting a SEBI-registered advisor for major decisions."""


def chat(
    memory: EntityMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """
    Send one user message to FinCoach with entity memory.
    Extraction runs before the API call — profile is current for THIS turn.
    """

    memory.add_message(role="user", content=user_message)
    # Adds to buffer + archive. Runs extraction (synchronous mode).
    # Profile is updated BEFORE the API call below.

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
        # Context: [system: persona] + [system: entity profile] + [recent buffer]
        # The entity profile is always current — updated in the add_message() call above.
    )

    assistant_reply = response.choices[0].message.content
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        completion = memory.entity_profile.completion_percentage()
        profile_tokens = memory.entity_profile.token_count()
        print(f"[Turn {memory._total_turns}] "
              f"API input: {response.usage.prompt_tokens} tokens | "
              f"Profile: {completion:.0%} complete ({profile_tokens} tokens)")

    return assistant_reply


print("FinCoach chat() with entity memory defined.")

FinCoach chat() with entity memory defined.


---
## Part 6 — Demo: Profile Building Turn by Turn

Watch the entity profile fill in as Chiru shares information. After each turn, the profile update is printed — which fields changed, from what, to what.

In [10]:
entity_memory = EntityMemory(
    session_id="session_em_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Expected extraction: name=Chiru, monthly_salary=₹1,20,000

    "My monthly expenses are ₹60,000 and I'm quite conservative with money.",
    # Expected: monthly_expenses=₹60,000, risk_profile=conservative

    "I'm 32 years old and work at Infosys in Mumbai. I have no dependents.",
    # Expected: age=32, employer=Infosys, location=Mumbai, dependents=0

    "I have an FD of ₹50,000 that matures in about 3 months. No other investments yet.",
    # Expected: fd_amount=₹50,000, fd_maturity_months=3

    "My goal is to retire at 55 and build a corpus of at least ₹2 crore.",
    # Expected: retirement_age=55, retirement_corpus_target=₹2 crore

    "I never want to invest in equity or stock markets — only fixed income.",
    # Expected: investment_constraints=never invest in equity or stock markets
    # This is the safety-critical field — it must NEVER be lost.

    "Based on my profile, what should I do with my FD when it matures?",
    # No new facts — tests whether FinCoach uses the complete profile.
    # Expected: advice that respects the equity constraint and conservative profile.
]

print("ENTITY MEMORY DEMO — Profile builds turn by turn")
print("Watch: [PROFILE] lines show exactly what was extracted and updated")
print("=" * 65)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=entity_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
entity_memory.print_stats()

[EntityMemory] Initialised — session: session_em_demo_001
  User: chiru_001 | Recent turns: 5
ENTITY MEMORY DEMO — Profile builds turn by turn
Watch: [PROFILE] lines show exactly what was extracted and updated

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
  [PROFILE] Updated: ['name', 'monthly_salary']
    name → Chiru
    monthly_salary → ₹1,20,000
[Turn 1] API input: 245 tokens | Profile: 10% complete (51 tokens)
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, you're in a good position to plan for savings and investments. It's generally advisable to allocate a portion of your income towards different financial goals such as emergency savings, retirement, and other investments. A common rule of thumb is the 50/30/20 rule, where 50% goes to necessities, 30% to discretionary spending, and 20% to savings and investments. How can I assist you further with your financial planning?

--- Turn 2 ---
User: My monthly expenses are ₹60,000 and I

---
## Part 7 — Demo: In-Place Update — The Salary Change

In [11]:
print("IN-PLACE UPDATE DEMO — Chiru gets a raise")
print("=" * 65)

print(f"\nProfile BEFORE update:")
print(f"  monthly_salary = {entity_memory.entity_profile._profile['monthly_salary']}")
print(f"  employer       = {entity_memory.entity_profile._profile['employer']}")

# Turn 8: Chiru changes jobs and gets a raise.
update_message = "I just changed jobs — I now work at TCS and my new salary is ₹1,50,000 per month."
print(f"\nUser: {update_message}")

response = chat(memory=entity_memory, user_message=update_message, verbose=True)
print(f"FinCoach: {response}")

print(f"\nProfile AFTER update:")
print(f"  monthly_salary = {entity_memory.entity_profile._profile['monthly_salary']}")
print(f"  employer       = {entity_memory.entity_profile._profile['employer']}")

print(f"\nUpdate history (last 2 entries):")
for entry in entity_memory.entity_profile._update_history[-2:]:
    print(f"  Field: {entry['field']}")
    print(f"    Old: {entry['old_value']}")
    print(f"    New: {entry['new_value']}")
    print(f"    At : {entry['timestamp'][:19]}Z")

print()
print("KEY POINT:")
print("  Old salary  (₹1,20,000) → REPLACED by new salary (₹1,50,000)")
print("  Old employer (Infosys)  → REPLACED by new employer (TCS)")
print("  No ambiguity. No stale facts. The profile is always current.")
print("  Compare with Technique 06: BOTH old and new would be in ChromaDB.")
print("  The model would have to decide which is current. Here it just reads the profile.")

IN-PLACE UPDATE DEMO — Chiru gets a raise

Profile BEFORE update:
  monthly_salary = ₹1,20,000
  employer       = Infosys

User: I just changed jobs — I now work at TCS and my new salary is ₹1,50,000 per month.
  [PROFILE] Updated: ['employer', 'monthly_salary']
    employer → TCS
    monthly_salary → ₹1,50,000
[Turn 8] API input: 1051 tokens | Profile: 75% complete (172 tokens)
FinCoach: With your new position at TCS and an increased monthly salary of ₹1,50,000, you have additional resources to allocate towards your financial goals. Consider directing a portion of this salary increase towards enhancing your savings and investments, particularly in fixed income options that suit your conservative risk profile. This could include increasing contributions to existing FDs, exploring higher-yielding debt instruments, or setting up a systematic investment plan in a debt mutual fund. As your income has changed, it's a good time to reassess your budget and savings plan to ensure they align wi

---
## Part 8 — Cross-Session Persistence: The Returning User

In [12]:
SESSION_FILE = "/tmp/fincoach_em_session_demo_001.json"

entity_memory.persist(SESSION_FILE)
# Saves: full entity profile (all fields + update history) + recent buffer + archive.

print("\n--- Simulating new session (returning user) ---\n")

restored_memory = EntityMemory.load(SESSION_FILE)
restored_memory.system_prompt = FINCOACH_SYSTEM_PROMPT
# Restore the system prompt — it is agent config, not user data.

# Create a FRESH session — empty buffer.
restored_memory.session_id = "session_em_002"
restored_memory._recent_buffer = []  # Clear the buffer — new session starts fresh.
# The ENTITY PROFILE survives — this is the key.
# Chiru does not need to re-introduce himself.

print("New session started. Buffer is empty. Profile is loaded from previous session.")
print(f"Profile completion: {restored_memory.entity_profile.completion_percentage():.0%}")

# Chiru asks a question that requires facts from the previous session.
follow_up = "Hi, I'm back. Given my new salary at TCS, should I increase my SIP amount?"
# Tests: does FinCoach know the new salary (₹1,50,000) from the persisted profile?
print(f"\nUser: {follow_up}")

response = chat(memory=restored_memory, user_message=follow_up, verbose=True)
print(f"FinCoach: {response}")

print()
print("Expected: FinCoach references ₹1,50,000 (new TCS salary) without being told.")
print("The entity profile persisted across the session boundary.")

[EntityMemory] Persisted — profile 75% complete

--- Simulating new session (returning user) ---

[EntityMemory] Initialised — session: session_em_demo_001
  User: chiru_001 | Recent turns: 5
[EntityMemory] Loaded — profile 75% complete
New session started. Buffer is empty. Profile is loaded from previous session.
Profile completion: 75%

User: Hi, I'm back. Given my new salary at TCS, should I increase my SIP amount?
  [PROFILE] No new facts extracted
[Turn 9] API input: 367 tokens | Profile: 75% complete (172 tokens)
FinCoach: Hi Chiru! Welcome back. It seems there might be a recent change in your salary, as your profile indicates a monthly salary of ₹1,50,000. If this has increased, it's a good opportunity to reassess your savings and investment strategy.

Since you prefer fixed income instruments and have a conservative risk profile, consider allocating any additional disposable income to instruments like recurring deposits or debt mutual funds (if you haven't explored them yet), w

---
## Part 9 — Entity Memory + Vector Store: The Production Combination

In [13]:
# Illustrative — shows how Entity Memory and Vector Store fit together.

print("ENTITY MEMORY + VECTOR STORE — The Production Combination")
print("=" * 65)

print("""
EVERY API CALL RECEIVES:
────────────────────────
[system: FinCoach persona]              ~130 tokens  (constant)
     +
[system: entity profile]                ~150 tokens  (always current, no search needed)
  USER PROFILE for chiru_001:
  [PERSONAL] name: Chiru, age: 32, employer: TCS, location: Mumbai
  [FINANCIAL] salary: ₹1,50,000, expenses: ₹60,000, risk: conservative
  [ASSETS] fd_amount: ₹50,000, fd_maturity: 3 months
  [GOALS] retirement_age: 55, corpus: ₹2 crore
  [PREFERENCES] investment_constraints: never invest in equity
     +
[system: retrieved memories (vector store)]  ~200 tokens  (semantic search, past sessions)
  [2025-03-01] user: I'm worried about market volatility affecting my debt funds
  [2025-04-15] user: I opened a PPF account last month with ₹500/month contribution
     +
[recent buffer]                              ~500 tokens  (current session verbatim)
─────────────────────────────────────────────────────────────
TOTAL:                                       ~980 tokens   (predictable ceiling)


WHAT EACH LAYER ANSWERS:
─────────────────────────
Entity Profile    → "Who is this user?" (structured, current, always injected)
Vector Store      → "What did we discuss?" (semantic, historical, retrieved by relevance)
Recent Buffer     → "What are we talking about right now?" (verbatim, current session)


WHAT THE COMBINED SYSTEM SOLVES:
─────────────────────────────────
✅ Cross-session: profile + vector store both persist across sessions
✅ No stale facts: profile is always current (in-place updates)
✅ Fuzzy recall: vector store finds semantically relevant past conversations
✅ Structured recall: profile answers structured fact questions directly
✅ Multi-tenant: both layers filter by user_id
✅ Predictable cost: profile tokens (fixed) + retrieval tokens (bounded) + buffer (bounded)


WHAT IS STILL MISSING (→ Technique 08):
─────────────────────────────────────────
❌ Relationships: profile knows salary and FD separately but not that FD → matures into → investment capital
❌ Entity connections: who recommended which product, which goal depends on which asset
❌ Temporal logic: when did salary change? What was it before? (→ Technique 24 Graphiti)
These require a Knowledge Graph — that is Technique 08.
""")

ENTITY MEMORY + VECTOR STORE — The Production Combination

EVERY API CALL RECEIVES:
────────────────────────
[system: FinCoach persona]              ~130 tokens  (constant)
     +
[system: entity profile]                ~150 tokens  (always current, no search needed)
  USER PROFILE for chiru_001:
  [PERSONAL] name: Chiru, age: 32, employer: TCS, location: Mumbai
  [FINANCIAL] salary: ₹1,50,000, expenses: ₹60,000, risk: conservative
  [ASSETS] fd_amount: ₹50,000, fd_maturity: 3 months
  [GOALS] retirement_age: 55, corpus: ₹2 crore
  [PREFERENCES] investment_constraints: never invest in equity
     +
[system: retrieved memories (vector store)]  ~200 tokens  (semantic search, past sessions)
  [2025-03-01] user: I'm worried about market volatility affecting my debt funds
  [2025-04-15] user: I opened a PPF account last month with ₹500/month contribution
     +
[recent buffer]                              ~500 tokens  (current session verbatim)
──────────────────────────────────────────────

---
## Key Takeaways

**What you built:** An `EntityMemory` system with a schema-driven `EntityProfile`, an LLM-powered extractor using JSON response mode, in-place profile updates with full audit history, and a persistence layer that survives session boundaries.

---

### The three things to remember

1. **In-place update is the defining feature.** When Chiru changes jobs, `profile["employer"]` is replaced — not appended. There is always exactly one current value per field. This is the fundamental improvement over vector stores, where both old and new employer would be stored and retrieved simultaneously, leaving the model to guess which is current.

2. **The schema is the most important design decision.** Every field you include costs tokens on every call. Every field you omit may force the agent to ask the user again or guess. Design the schema for your domain before writing any code. For FinCoach, `investment_constraints` is the most safety-critical field — a user who says "never invest in equity" must have that constraint persist forever, across all sessions, across all compression cycles.

3. **Use `response_format={"type": "json_object"}` for extraction.** This forces GPT-4o-mini to return valid JSON — no markdown fences, no preamble, no parsing headaches. Combined with `temperature=0.0`, extraction becomes deterministic and reliable enough for production financial data handling.



